# Oppitunti 10 - AI-agentit tuotannossa

Tässä oppitunnissa opit **tuotantokäytännöt** AI-agenteille käyttäen Microsoft Agent Frameworkia (Python). Käsittelemme:

- **Havaittavuus** — ajanseurannan ja lokituksen lisääminen agenttien vuorovaikutuksiin
- **Arviointi** — arvioija-agentin käyttäminen vastausten laadun pisteyttämiseen
- **Kustannusten hallinta** — strategioita tokenien optimointiin ja mallin valintaan

Skenaario on **matka-agentti**, joka auttaa käyttäjiä suunnittelemaan matkoja, ja johon on lisätty monitorointi ja arviointi.


## Asennus


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Tuotantoon liittyvät näkökohdat

Tekoälyagenttien siirtäminen prototyypeistä tuotantoon vaatii huolellista huomiota kolmeen pilariin:

1. **Havaittavuus** — Tarvitset näkyvyyden siihen, mitä agentti tekee, kuinka kauan se vie, ja mitä työkaluja se kutsuu. Ilman seurantaa ja lokitusta tuotanto-ongelmien vianmääritys on lähes mahdotonta.

2. **Arviointi** — Automaattiset laatutarkistukset varmistavat, että agentin vastaukset pysyvät ajan myötä tarkkoina, täydellisinä ja hyödyllisinä. Arvioiva agentti voi pisteyttää vastauksia määriteltyjen kriteerien mukaan.

3. **Kustannusten hallinta** — Tokenien käyttö vaikuttaa suoraan kustannuksiin. Strategiat kuten kehotteiden optimointi, mallin valinta ja välimuisti auttavat pitämään kulut hallinnassa ilman laadusta tinkimistä.


## Havainnoitavan agentin rakentaminen

Määrittelemme matkustustyökalut ja kääritään agenttikutsu ajoituksen ympärille, jotta voimme seurata latenssia. Tuotannossa integroisit OpenTelemetryn tai vastaavan jäljitystaustan kanssa.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Arviointimallit

Yleinen tuotantokäytäntö on käyttää toista agenttia **arvioijana**. Arvioija pisteyttää ensisijaisen agentin vastauksen ennalta määriteltyjen kriteerien, kuten täydellisyyden, tarkkuuden ja hyödyllisyyden, perusteella.

Tämä mahdollistaa:
- Automaattiset laatukynnykset ennen kuin vastaukset saavuttavat käyttäjät
- Regressioiden havaitseminen, kun kehotteet tai mallit muuttuvat
- Agentin suorituskyvyn jatkuva seuranta ajan kuluessa


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Kustannusten hallintastrategiat

Kustannusten hallinta on kriittistä tuotantokäytössä oleville tekoälyagenteille. Tässä ovat keskeiset strategiat:

| Strategia | Kuvaus |
|---|---|
| **Kehoteoptimointi** | Pidä järjestelmäohjeet ytimekkäinä. Poista päällekkäinen konteksti vähentääksesi syötteen tokeneja. |
| **Mallin valinta** | Käytä pienempiä, edullisempia malleja (esim. GPT-4o-mini) yksinkertaisiin tehtäviin, kuten luokitteluun tai tietojen poimintaan, ja varaa suuremmat mallit monimutkaiseen päättelyyn. |
| **Välimuisti** | Välimuistita työkalujen tulokset ja toistuvat kyselyt välttääksesi päällekkäiset API-kutsut. |
| **Token-rajoitukset** | Aseta `max_tokens`-rajat estääksesi odottamattoman pitkiä vastauksia. |
| **Ryhmittely** | Ryhmittele useita käyttäjäkyselyjä yhdeksi API-kutsuksi, kun mahdollista. |

Käytännössä monitasoinen lähestymistapa toimii hyvin: ohjaa suoraviivaiset pyynnöt nopealle, edulliselle mallille ja siirrä vain monimutkaiset kyselyt kykenevämmälle mallille.


## Yhteenveto

Tässä oppitunnissa opit, miten:

1. **Lisätä havaittavuutta** agenttien vuorovaikutukseen ajoituksen ja lokituksen avulla, luoden perustan jäljitykselle ja valvonnalle.
2. **Arvioida agenttien vastauksia** automaattisesti käyttämällä arvioija-agenttia, joka pisteyttää täydellisyyden, tarkkuuden ja hyödyllisyyden.
3. **Hallita kustannuksia** kehotteiden optimoinnin, mallin valinnan, välimuistin käytön ja token-budjettien avulla.

Nämä tuotantokäytännöt auttavat varmistamaan, että tekoälyagenttisi ovat luotettavia, mitattavia ja laajassa mittakaavassa kustannustehokkaita.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, huomioithan, että automaattiset käännökset voivat sisältää virheitä tai epätarkkuuksia. Alkuperäistä asiakirjaa sen alkuperäiskielellä tulee pitää määräävänä lähteenä. Tärkeiden tietojen osalta suosittelemme ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai virheellisistä tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
